## What this analysis can tell us

**This is a descriptive comparison, not evidence that Stephen Watson or burglary attendance caused a change.** The official data contain a major GMP recording break before his appointment. A synthetic control can be calculated, but it cannot repair that break or supply a credible causal estimate.

The question follows my [earlier post on burglary attendance](https://andreasthinks.me/posts/burglary_attendance/): how did recorded residential burglary outcomes change around Watson taking office on 24 May 2021? April–June 2021 is a transition quarter; July–September 2021 is the first full quarter afterwards. These dates are the research brief's intervention definition. National attendance requirements, other reforms, pandemic disruption and changes in crime recording prevent a clean attribution to one person or policy.

I use the Home Office's offence-cohort outcomes, through **April–June 2026**. These files cannot replicate an August 2026 claim. The latest workbook is dated 22 October 2026, but its crimes stop in June. A publication date is not an offence date.

In [1]:
from pathlib import Path
import json
import pandas as pd
pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
from IPython.display import display, Markdown
from analysis import run, BASE, GMP, PEERS
from figures import make_figures
panel, benchmarks, model, results = run()
make_figures(panel, benchmarks, model)
def table(filename, columns=None):
    frame = pd.read_csv(BASE / 'data' / filename)
    display(frame[columns] if columns else frame)
summary = pd.read_csv(BASE / 'data/period_summary.csv')
g = summary[summary.force == GMP].set_index('period')
before, recent = g.loc['Before: 2020/21'], g.loc['2024/25']
display(Markdown(f"For crimes recorded in 2020/21, GMP's charge/summons rate is **{before.rate:.2f}%** ({before.charge:,.0f} of {before.denominator:,.0f}). In 2024/25 it is **{recent.rate:.2f}%** ({recent.charge:,.0f} of {recent.denominator:,.0f}), a change of **{recent.rate-before.rate:+.2f} percentage points**. Both are the outcomes known at their respective workbook snapshots, not measurements at equal follow-up ages."))

For crimes recorded in 2020/21, GMP's charge/summons rate is **3.33%** (502 of 15,056). In 2024/25 it is **8.42%** (864 of 10,267), a change of **+5.08 percentage points**. Both are the outcomes known at their respective workbook snapshots, not measurements at equal follow-up ages.

## What is counted

The numerator is **outcome 1, Charged/Summonsed**. The denominator is the sum across all outcome types, including code 0, **Not yet assigned an outcome**, for the selected offences recorded in that quarter. This is a count of recorded crimes, not victims, suspects, convictions or investigations closed that quarter. A charge is not a conviction. Code 0 is not synonymous with every unsolved crime: a closed case without a suspect has a different outcome code.

The other numeric column in the files counts investigations closed in the quarter. Dividing that flow by new crimes in the same quarter mixes different cohorts. It is retained in the extraction for audit but never used in these rates. “Detection rate” below means the specified cohort charge/summons percentage, not an unspecified solved measure.

Residential burglary includes attempts, distraction burglary and aggravated burglary. From April 2017 to March 2023 I use codes 28E–28H and 29A. From April 2023 I use 28I–28P, 29B and 29C, adding **homes and unconnected buildings together**. The latest workbook notes describe the split. Earlier domestic burglary is retained in the downloadable audit using 28A–28D and 29, but excluded from model fitting because the April 2017 classification widened the residential boundary. Old residential codes still contain 2,673 crimes in 2023/24 nationally. I include those residual 28E–28H/29A counts alongside the new codes after April 2023, because they remain residential crimes in these offence cohorts. The residual counts are audited separately; old domestic codes are not carried into the residential definition.

A secondary “broader disposal” measure sums codes **1, 2, 3, 4, 6, 7 and 8**: charges, youth/adult cautions, taken into consideration, penalty notices, cannabis/khat warnings and community resolutions. It excludes offender death, non-prosecution decisions, diversion code 22 and deferred prosecution codes 23–24. This fixed definition is a sensitivity check, **not a reconstruction of GMP's solved statistic**. The notes record changes in disposal practice and new deferred prosecution codes in April 2026.

Definitions and caveats are transcribed from the cached workbooks' headers, outcome labels and Notes sheets. See the [official data collection](https://www.gov.uk/government/statistical-data-sets/police-recorded-crime-and-outcomes-open-data-tables) and [official user guide](https://www.gov.uk/government/statistics/police-recorded-crime-open-data-tables/police-recorded-crime-and-outcomes-open-data-tables-user-guide). The user guide and data collection page were independently consulted alongside the workbook notes.

## Counts, rates and real comparators

I chose the other metropolitan territorial forces as real comparators on institutional/geographic grounds, without selecting on their post-2021 performance: Merseyside, Metropolitan Police, Northumbria, South Yorkshire, West Midlands and West Yorkshire. These are contextual comparisons, not matched or randomly assigned controls. West Midlands has explicit reporting warnings, so the headline peer line excludes it; the full group and individual force results remain visible.

The national benchmark is **all 42 other territorial England and Wales forces**, subject to available valid cells each quarter, excluding GMP and British Transport Police. It is a burglary-count-weighted rate: total charges divided by total recorded residential burglaries, not the average force percentage. It is not an all-crime national charge rate, nor necessarily an exact reproduction of a published national headline. Workbook warnings still apply to the inclusive national benchmark. The tables expose the number of contributing forces; a separate quality-screened benchmark is supplied with the derived data.

The latest quarters have less time for outcomes to be assigned. The main model summary ends at March 2025 to reduce that problem, but this does **not** establish equal maturity. The descriptive table also shows 2025/26 and the newest quarter, with unresolved counts and proportions.

In [2]:
display(summary[summary.force.isin([GMP, 'Territorial England and Wales excluding GMP', 'Peer group excluding West Midlands'])].round(3))

,period,force,denominator,charge,unresolved,rate,unresolved_rate
0,Before: 2020/21,Greater Manchester,15056.0,502.0,2.0,3.334,0.013
5,Before: 2020/21,Peer group excluding West Midlands,71187.0,3109.0,555.0,4.367,0.780
7,Before: 2020/21,Territorial England and Wales excluding GMP,180940.0,8654.0,2078.0,4.783,1.148
10,First full year,Greater Manchester,18201.0,834.0,5.0,4.582,0.027
15,First full year,Peer group excluding West Midlands,67536.0,2880.0,402.0,4.264,0.595
17,First full year,Territorial England and Wales excluding GMP,177178.0,8508.0,1324.0,4.802,0.747
20,2024/25,Greater Manchester,10267.0,864.0,26.0,8.415,0.253
25,2024/25,Peer group excluding West Midlands,59952.0,2819.0,665.0,4.702,1.109
27,2024/25,Territorial England and Wales excluding GMP,156016.0,8951.0,1926.0,5.737,1.234
30,2025/26,Greater Manchester,8382.0,827.0,156.0,9.866,1.861


:::{.dark-fig}
![GMP counts and weighted comparison rates. Grey bands mark the excluded 2019/20 GMP financial year and the 2021 transition quarter. Each point is an offence cohort, with unequal snapshot follow-up.](figures/trends_dark.png)
:::

:::{.light-fig}
![GMP counts and weighted comparison rates. Grey bands mark the excluded 2019/20 GMP financial year and the 2021 transition quarter. Each point is an offence cohort, with unequal snapshot follow-up.](figures/trends_light.png)
:::

:::{.dark-fig}
![Real comparator force rates. West Midlands is shown for transparency despite the workbook comparability warning; it is excluded from synthetic-control donors.](figures/comparators_dark.png)
:::

:::{.light-fig}
![Real comparator force rates. West Midlands is shown for transparency despite the workbook comparability warning; it is excluded from synthetic-control donors.](figures/comparators_light.png)
:::

In [3]:
display(summary[summary.force.isin(PEERS)].round(3))
display(benchmarks[benchmarks.quarter >= pd.Period('2017Q2')][['quarter','force','n_forces','expected_forces']])

,period,force,denominator,charge,unresolved,rate,unresolved_rate
1,Before: 2020/21,Merseyside,5780.0,259.0,5.0,4.481,0.087
2,Before: 2020/21,Metropolitan Police,42816.0,1592.0,246.0,3.718,0.575
4,Before: 2020/21,Northumbria,4492.0,478.0,203.0,10.641,4.519
6,Before: 2020/21,South Yorkshire,7467.0,269.0,4.0,3.603,0.054
8,Before: 2020/21,West Midlands,13931.0,468.0,978.0,3.359,7.020
9,Before: 2020/21,West Yorkshire,10632.0,511.0,97.0,4.806,0.912
11,First full year,Merseyside,5011.0,215.0,14.0,4.291,0.279
12,First full year,Metropolitan Police,39930.0,1525.0,130.0,3.819,0.326
14,First full year,Northumbria,4508.0,336.0,23.0,7.453,0.510
16,First full year,South Yorkshire,8448.0,352.0,0.0,4.167,0.000


,quarter,force,n_forces,expected_forces
12,2017Q2,Territorial England and Wales excluding GMP,42,42
13,2017Q3,Territorial England and Wales excluding GMP,42,42
14,2017Q4,Territorial England and Wales excluding GMP,42,42
15,2018Q1,Territorial England and Wales excluding GMP,42,42
16,2018Q2,Territorial England and Wales excluding GMP,42,42
17,2018Q3,Territorial England and Wales excluding GMP,42,42
18,2018Q4,Territorial England and Wales excluding GMP,42,42
19,2019Q1,Territorial England and Wales excluding GMP,42,42
20,2019Q2,Territorial England and Wales excluding GMP,42,42
21,2019Q3,Territorial England and Wales excluding GMP,42,42


## The recording break is central

The 2019/20 Notes sheet says GMP could not supply July 2019–March 2020 data following a new IT system in July 2019, and separately says its cohort outcomes are not up to date or comparable. This is the iOPS-era break identified in the brief. I exclude **all of 2019/20**, including April–June 2019, from fitting and mark it as a gap in figures. The actual extracted cells remain below. Some source outcome-0 cells are negative, including South Yorkshire in 2017/18 and Dyfed-Powys in 2023/24. I preserve these signed source values in the summed denominator; they cannot literally be counts of open investigations. Their cause cannot be established from the cached notes. The negative-cell audit is retained, rates with non-positive denominators are unavailable, and a sensitivity specification makes every negative-cell force-quarter unavailable (excluding affected pre-period donors). No missing values are imputed as zeros and no line is interpolated across the missing model data.

After that exclusion, the comparable residential definition provides eight pre-break quarters (April 2017–March 2019) and just four quarters after the break and before the appointment (April 2020–March 2021). Those four also coincide with the pandemic. Numerical availability after the break does not certify consistent crime recording. Changes in what gets recorded can change both counts and rates even if investigative performance is unchanged.

In [4]:
display(panel[(panel.force == GMP) & panel.quarter.between(pd.Period('2018Q2'), pd.Period('2021Q2'))][['quarter','denominator','charge','unresolved','rate','missing_cells','recording_warning']])
table('out_of_era.csv')
negative = pd.read_csv(BASE/'data/negative_cells.csv')
display(negative.groupby(['financial_year','force']).agg(negative_cells=('value','size'),signed_total=('value','sum')))

,quarter,denominator,charge,unresolved,rate,missing_cells,recording_warning
718,2018Q2,5314.0,216.0,15.0,4.064735,0,False
762,2018Q3,5238.0,233.0,37.0,4.448263,0,False
806,2018Q4,5732.0,174.0,61.0,3.035590,0,False
850,2019Q1,5491.0,138.0,109.0,2.513203,0,False
894,2019Q2,4575.0,81.0,663.0,1.770492,0,True
938,2019Q3,NaN,NaN,NaN,NaN,110,True
982,2019Q4,NaN,NaN,NaN,NaN,110,True
1026,2020Q1,NaN,NaN,NaN,NaN,110,True
1070,2020Q2,2999.0,130.0,0.0,4.334778,0,False
1114,2020Q3,3954.0,150.0,0.0,3.793627,0,False


,financial_year,code,numeric_total,missing_cells
0,2023/24,28E,2008.0,0
1,2023/24,28F,507.0,0
2,2023/24,28G,29.0,0
3,2023/24,28H,2.0,0
4,2023/24,29A,127.0,0
5,2024/25,28E,3.0,0
6,2024/25,29A,11.0,0
7,2025/26,28F,1.0,0
8,2026/27,28E,1.0,0


negative_cells  signed_total
financial_year force                                            
2014/15        Avon and Somerset                 1          -1.0
               Bedfordshire                      3          -8.0
               Cambridgeshire                    7         -16.0
               Cleveland                         4         -20.0
               Devon and Cornwall                2          -2.0
               Dorset                            8         -10.0
               Durham                            5         -15.0
               Essex                             9         -41.0
               Gloucestershire                   7          -8.0
               Gwent                             3          -5.0
               Hampshire                         5         -19.0
               Hertfordshire                     7        -374.0
               Humberside                        3          -4.0
               Kent                              1          -1.0
               London, City of                   3          -3.0
               Merseyside                        4         -11.0
               Metropolitan Police               3        -158.0
               Norfolk                           3          -3.0
               North Yorkshire                   7         -12.0
               Northamptonshire                  3          -6.0
               Northumbria                       3         -11.0
               Nottinghamshire                   4         -20.0
               South Wales                       5         -65.0
               South Yorkshire                   7         -66.0
               Staffordshire                     1          -1.0
               Suffolk                           4         -13.0
               Surrey                            7         -24.0
               Warwickshire                      3          -8.0
               West Mercia                       8         -17.0
               West Midlands                     4          -5.0
               West Yorkshire                    2          -2.0
               Wiltshire                         5          -9.0
2015/16        Derbyshire                        1          -1.0
               Humberside                        5         -18.0
               Leicestershire                    3         -16.0
               London, City of                   1          -1.0
               South Yorkshire                   4          -7.0
2016/17        Essex                             1          -6.0
               Humberside                        3          -5.0
               South Yorkshire                   1          -3.0
               Wiltshire                         2          -6.0
2017/18        South Yorkshire                   4          -6.0
2023/24        Dyfed-Powys                       4         -26.0
               Humberside                        2          -4.0
2024/25        Humberside                        4         -12.0
2025/26        Humberside                        4         -14.0
2026/27        Humberside                        1          -4.0

## A synthetic control, with a failed evidence gate

A synthetic control is a weighted mixture of other forces. Weights must be nonnegative and sum to one. I minimise the unweighted mean squared difference between quarterly charge/summons percentages, using only the pre-intervention quarters. There is no intercept, outcome adjustment, post-period weight tuning or demographic matching.

All other forces are considered. British Transport Police is excluded because it is non-territorial. The Notes sheets identify stale/incomplete reporting for Dorset, Devon and Cornwall, Humberside, West Midlands and Leicestershire; these are excluded conservatively. Any remaining force with a missing pre-period rate or zero denominator is also excluded. Post-period performance is not an eligibility criterion. City of London is retained despite its small residential counts; one zero-denominator post quarter is unavailable. Predictions require only donors with positive fitted weights (weights below 1e-8 are set to zero and renormalised). All placebo ratios use the same complete post quarters across their target forces; their `n_post` column records that denominator. The complete exclusion and weight tables follow.

Before interpreting an effect, this analysis requires at least eight consecutive quarters of reliable pre-treatment data under the same recording regime, with at least four held out, and holdout RMSE no worse than 1 percentage point and twice the training RMSE (with a 0.1-point floor). These are conservative analysis rules, not established statistical cutoffs or a preregistration. **GMP fails the recording-continuity requirement regardless of the fitted numbers.** There are only four post-break quarters. A good fit across the break would not repair that failure.

For the main numerical diagnostic I nevertheless fit all 12 available pre quarters and retain the results. The chronological holdout trains on the eight pre-break quarters and predicts the four 2020/21 quarters. It directly tests transport across the recording break, but cannot independently validate the recording regime. A separate short-window specification trains on two quarters and holds out two; this is far too short for causal confidence.

In [5]:
table('donor_exclusions.csv')
table('weights.csv')
table('specifications.csv')
display(Markdown(f"The main fit uses **{results['primary']['donors']} donors**. Pre-period RMSE is **{results['primary']['pre_rmse']:.3f} percentage points**; chronological holdout RMSE is **{results['primary']['holdout_rmse']:.3f} points**. The mean July 2021–March 2025 gap is **{results['primary']['post_mean_gap']:+.3f} points**. This is an observed-minus-fitted difference, **not an estimated causal effect**."))

,force,reason
0,British Transport Police,Non-territorial transport force
1,Devon and Cornwall,2020/21 notes: cohort outcomes not comparable
2,Dorset,2018/19 notes: last refreshed January 2022
3,Humberside,2020/21 and later notes: cohort outcomes not comparable
4,Leicestershire,"2020/21 notes: missing outcome 22, denominator concern"
5,West Midlands,2019/20 and 2020/21 notes: cohort outcomes not comparable


,force,weight
0,Metropolitan Police,0.580799
1,Surrey,0.189542
2,Hertfordshire,0.142271
3,Wiltshire,0.071803
4,Hampshire,0.015586
5,Warwickshire,0.000000
6,Thames Valley,0.000000
7,Sussex,0.000000
8,West Mercia,0.000000
9,Suffolk,0.000000


,metric,start,end,n_pre,n_train,n_holdout,donors,pre_rmse,holdout_rmse,training_rmse,equal_weight_pre_rmse,post_mean_gap,post_rmse,n_post
0,rate,2017Q2,2021Q1,12,8,4,37,0.321706,0.481680,2.822420e-01,2.254974,2.224754,2.642857,15
1,rate,2017Q2,2019Q1,8,8,0,37,0.282242,NaN,NaN,2.311281,1.867929,2.327445,15
2,rate,2020Q2,2021Q1,4,2,2,37,0.000006,0.800956,7.857032e-07,2.137914,1.915277,2.254230,14
3,broad_rate,2017Q2,2021Q1,12,8,4,37,0.332327,0.518222,3.077425e-01,2.664484,2.636984,3.027520,15
4,rate (negative-cell quarters unavailable),2017Q2,2021Q1,12,8,4,36,0.321706,0.481613,2.822418e-01,2.297768,2.224827,2.642891,15


The main fit uses **37 donors**. Pre-period RMSE is **0.322 percentage points**; chronological holdout RMSE is **0.482 points**. The mean July 2021–March 2025 gap is **+2.225 points**. This is an observed-minus-fitted difference, **not an estimated causal effect**.

:::{.dark-fig}
![The actual constrained fit and its gap. The training period crosses a known recording break, so the gap has no secure causal interpretation.](figures/synthetic_dark.png)
:::

:::{.light-fig}
![The actual constrained fit and its gap. The training period crosses a known recording break, so the gap has no secure causal interpretation.](figures/synthetic_light.png)
:::

## How fragile is the comparison?

The specification table contrasts the full available pre-period, pre-break only, post-break only, and the broader disposal outcome. Its equal-weight pre-RMSE is an unoptimised benchmark. All fits use raw quarterly rates and the same documented donor-quality exclusions. Different windows change what is learned; none removes the data problem. With more donors than time points, especially in the two-quarter training window, optimal weights can be non-unique. Near-perfect short-window prefit is not evidence of reliability.

I refit after omitting **every eligible donor one at a time**, including donors with zero original weight. For in-space placebos, each eligible donor becomes the target and is fitted from the remaining donors; GMP is never a donor in those placebo fits. The post/pre RMSE ratio scales gaps by prefit quality. I show all results, then a screen retaining pre-RMSE at most twice GMP's. Rankings include GMP and their denominators. Near-zero prefit can inflate ratios. These are descriptive diagnostics, **not permutation p-values**: forces are not exchangeable or randomly selected for treatment.

In [6]:
table('leave_one_donor_out.csv')
table('placebos.csv')
p = results['placebo_gmp']
display(Markdown(f"GMP's fit-normalised rank is **{p['rank_all']:.0f} of {results['placebo_total']}** including GMP, or **{p['rank_screened']:.0f} of {results['placebo_screened']}** after the stated prefit screen. Leave-one-donor-out mean gaps range from **{results['loo_min']:+.3f} to {results['loo_max']:+.3f} percentage points**. These ranges measure specification sensitivity, not a confidence interval."))
table('post_window_sensitivity.csv')

,omitted,pre_rmse,post_mean_gap,n_post
0,Avon and Somerset,0.321709,2.225459,15
1,Bedfordshire,0.321706,2.224854,15
2,Cambridgeshire,0.321706,2.224851,15
3,Cheshire,0.321706,2.224758,15
4,Cleveland,0.321706,2.224753,15
5,Cumbria,0.321706,2.224782,15
6,Derbyshire,0.321707,2.224686,15
7,Durham,0.321706,2.224693,15
8,Dyfed-Powys,0.321706,2.224668,15
9,Essex,0.321707,2.224404,15


,force,pre_rmse,post_rmse,ratio,post_mean_gap,n_post,passes_prefit_screen,rank_all,rank_screened
0,Greater Manchester,0.321706,2.713687,8.435290,2.291275,14,True,1.0,1.0
1,Avon and Somerset,0.468791,2.137407,4.559399,-1.482262,14,True,11.0,11.0
2,Bedfordshire,0.966738,2.532137,2.619259,1.554304,14,False,26.0,NaN
3,Cambridgeshire,1.114850,2.476494,2.221370,1.742596,14,False,28.0,NaN
4,Cheshire,0.740693,1.402511,1.893512,0.208575,14,False,34.0,NaN
5,Cleveland,0.633726,1.242803,1.961105,0.061256,14,True,32.0,19.0
6,Cumbria,1.585200,4.372354,2.758235,1.906411,14,False,23.0,NaN
7,Derbyshire,0.681919,1.237875,1.815282,0.162940,14,False,35.0,NaN
8,Durham,1.241317,3.280817,2.643012,2.279522,14,False,25.0,NaN
9,Dyfed-Powys,2.240757,1.866928,0.833168,-1.256594,14,False,37.0,NaN


GMP's fit-normalised rank is **1 of 38** including GMP, or **1 of 19** after the stated prefit screen. Leave-one-donor-out mean gaps range from **+1.430 to +2.472 percentage points**. These ranges measure specification sensitivity, not a confidence interval.

,start,end,n,mean_gap
0,2021Q3,2023Q1,7,1.029170
1,2021Q3,2025Q1,15,2.224754
2,2021Q3,2026Q1,19,3.023758
3,2021Q3,2026Q2,20,3.159623


:::{.dark-fig}
![Post/pre RMSE ratios for GMP and all eligible placebo targets. Ranking does not establish exchangeability or causality.](figures/placebos_dark.png)
:::

:::{.light-fig}
![Post/pre RMSE ratios for GMP and all eligible placebo targets. Ranking does not establish exchangeability or causality.](figures/placebos_light.png)
:::

## Source snapshots and reproducibility

The 13 XLSX files supplied in `source-cache/` were downloaded by the host from official open-data links on 2 November 2026. Every supplied workbook is inspected, including the older domestic-burglary years. The manifest records filenames, SHA-256 hashes, workbook update text, coverage and headers. File naming dates are not assumed to be extraction dates: for example, the 2014/15 workbook says its data are as published in January 2019, despite its later filename. A Notes update date also need not be the force's last data refresh.

Only one supplied snapshot exists for each financial year. Therefore I cannot construct a panel measured a fixed number of months after every cohort, audit revisions between releases, or claim equal maturity across forces. The extracted unresolved proportions help expose this issue but do not correct it. Archived repeated releases would be needed for a consistent-age analysis.

The manifest records the direct attachment URLs verified against the Home Office collection page as well as the filenames and hashes. The [2025/26 technical annex](https://www.gov.uk/government/statistics/crime-outcomes-in-england-and-wales-2025-to-2026/crime-outcomes-in-england-and-wales-technical-annex), linked in workbook notes, is an additional documentary reference for independent review, not a page inspected here.

The compact extracted public data, code and audit outputs live beside this notebook. Full XLSX downloads are not included. `extract.py` rebuilds the data from those workbooks; `analysis.py` rebuilds every table and `figures.py` both theme variants. `build_notebook.py` generates this notebook; it was executed through a Jupyter kernel on the host. The original sandbox could not bind kernel sockets, so `execute.py` provides an in-process alternative. The tests cover weighted rates, missingness, time splits, constrained weights, outcome mapping, source totals and absence of post-treatment fitting leakage. See `_README.md` for exact commands and file roles.

In [7]:
manifest = json.loads((BASE/'data/manifest.json').read_text())
display(pd.DataFrame(manifest)[['file','notes_update','years','quarters','burglary_rows','sha256']])
import unittest
suite = unittest.defaultTestLoader.discover(str(BASE), pattern='test_analysis.py')
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Analysis checks failed' 

,file,notes_update,years,quarters,burglary_rows,sha256
0,prc-outcomes-open-data-jun2026-tables-221026.xlsx,Updated: 22 October 2026,[2026/27],[1],28512,4e79a0ebe4f12011d320d1390529df9541f8860464a483f7f9f643df3ffcc8ab
1,prc-outcomes-open-data-mar2015-tables-241024.xlsx,Updated: 24 January 2019,[2014/15],"[1, 2, 3, 4]",30096,cf651dd5aa388e8e11ee3ffe9af76828dea270c795ec0d447ed73b6140079c12
2,prc-outcomes-open-data-mar2016-tables-240724.xlsx,Updated: 27 April 2023,[2015/16],"[1, 2, 3, 4]",34848,2ce603e1c17f902d6e58447fbedbe61d7b8c02aec4d11772b4ec522693e5f481
3,prc-outcomes-open-data-mar2017-tables-240724.xlsx,Updated 19th October 2023,[2016/17],"[1, 2, 3, 4]",65120,b8971b23222e1aa3245229f568bd5864e4903a8a5c76cdd5744a7fe22243edca
4,prc-outcomes-open-data-mar2018-tables-240724.xlsx,Updated: 19th October 2023,[2017/18],"[1, 2, 3, 4]",65824,4b04cb277bac29ae7d9e0d178da1433d50a4d8ab6ef902a5189dff8f7833b513
5,prc-outcomes-open-data-mar2019-tables-240724.xlsx,Updated 25th January 2024,[2018/19],"[1, 2, 3, 4]",65824,4938d7e5bd8955cae81e65f640a42b89fda180d7e72dc2455ab5a1118ceb6b0d
6,prc-outcomes-open-data-mar2020-tables-240724.xlsx,Updated: 25th January 2024,[2019/20],"[1, 2, 3, 4]",65824,e02fc76eb0c980ef357d6c5cb8a6cac555c16529360482330cb4314a4dc2e9cf
7,prc-outcomes-open-data-mar2021-tables-240425.xlsx,Updated: 24 April 2025,[2020/21],"[1, 2, 3, 4]",104544,c87feca5b7aab1ae1573fd0267e99cfd9b36b93f647cf4ccb5c2c459fa895fdf
8,prc-outcomes-open-data-mar2022-tables-240725.xlsx,Updated: 4 August 2025,[2021/22],"[1, 2, 3, 4]",104544,7536a78f8cb8416e14246cb2811253f0c3844d5d1a3fb511c4b37b4991144dbc
9,prc-outcomes-open-data-mar2023-tables-230726.xlsx,Updated: 23 July 2026,[2022/23],"[1, 2, 3, 4]",104544,590d7ecfc1ad98e445494094095f762b2cd4b2a131a0862eac484d592ddf063a


test_mappings (test_analysis.AnalysisTests.test_mappings) ... 

ok


test_missing_not_zero (test_analysis.AnalysisTests.test_missing_not_zero) ... 

ok


test_no_post_leakage (test_analysis.AnalysisTests.test_no_post_leakage) ... 

... (8 outputs left out)

ok


test_weighted_rate (test_analysis.AnalysisTests.test_weighted_rate) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 3.106s

OK


## What I would conclude

The observed counts and charge/summons proportions are useful descriptions of what these snapshots contain. They do not isolate the effect of a chief constable, attendance policy or any single reform. The national mandate and bundled changes contaminate comparisons; non-random force selection, recording practices, classification changes, pandemic conditions, differing follow-up and ongoing data revisions all remain plausible explanations for part of any gap.

The synthetic control is retained because failed diagnostics should be visible. It should not be presented as a credible counterfactual, and its gap should not be converted into “extra crimes solved”. Stronger attribution would require a defensible stable pre-period, comparable outcome maturity, and a design that separates simultaneous changes.

**AI assistance:** Codex helped extract the supplied public workbooks, write and execute the analysis, produce diagnostics and draft this text. Independent human review of the data mapping, provenance and interpretation is required before publication. This post remains an unlinked draft with `noindex`.